## Submit a VeRL RL post-training job to the Ray cluster

In this section, we will submit a reinforcement learning (RL) post-training job to our Ray cluster using [VeRL](https://github.com/verl-project/verl), a framework for RL-based training of large language models.

After you finish this section,

-   you should understand how RL post-training works at a high level (generate responses, score them, update the model)
-   and you should be able to submit a VeRL GRPO training job to a Ray cluster using `ray job submit`.

### Background: RL post-training with GRPO

In standard fine-tuning (supervised), you show the model correct answers and it learns from them directly. In RL post-training, the process is different:

1.  Give the model a prompt (e.g., a math question)
2.  Let the model **generate** multiple responses
3.  **Score** each response with a reward function (e.g., did it get the correct answer?)
4.  **Update** the model to produce more high-scoring responses

GRPO (Group Relative Policy Optimization) is an RL algorithm that compares responses within a group — responses that scored above the group average are reinforced, and those below are discouraged. Unlike PPO, GRPO does not require a separate critic model, which reduces GPU memory requirements.

VeRL uses Ray internally to coordinate the different components of RL training: the actor model (being trained), the rollout engine (generating responses), the reference policy (for KL penalty), and the reward computation.

### Start the Ray cluster

First, we need to start the Ray cluster. The VeRL dependencies (including the `verl` package) are already installed in the Ray worker image.

On the host (node-mltrain), build the Ray worker image (if you haven't already) and start the cluster:

``` bash
# run on node-mltrain
docker build -t ray-rocm:2.42.1 -f mltrain-chi/docker/Dockerfile.ray-rocm .
```

``` bash
# run on node-mltrain
export HOST_IP=$(curl --silent http://169.254.169.254/latest/meta-data/public-ipv4)
docker compose -f mltrain-chi/docker/docker-compose-ray-rocm.yaml up -d
```

Then start the Jupyter submission container, mounting the `workspace_verl` directory:

``` bash
# run on node-mltrain
docker build -t jupyter-ray -f mltrain-chi/docker/Dockerfile.jupyter-ray .
HOST_IP=$(curl --silent http://169.254.169.254/latest/meta-data/public-ipv4)
docker run -d --rm -p 8888:8888 \
  -v ~/mltrain-chi/workspace_verl:/home/jovyan/work/ \
  -e RAY_ADDRESS=http://${HOST_IP}:8265/ \
  --name jupyter jupyter-ray
```

Verify the cluster is running by visiting the Ray dashboard at `http://<HOST_IP>:8265`. You should see 2 worker nodes, each with 1 GPU.

### Prepare the GSM8K dataset

We will train on [GSM8K](https://huggingface.co/datasets/openai/gsm8k), a dataset of grade school math problems. VeRL expects data in parquet format with a specific schema.

Open a terminal inside the Jupyter environment ("File > New > New Terminal") and install the `datasets` library, then run the data preparation script:

``` bash
# run in a terminal inside jupyter container
cd ~/work
pip install datasets
python prepare_data.py --local_save_dir data/gsm8k --max_train_samples 200 --max_test_samples 100
```

This downloads a small subset of GSM8K (200 training, 100 test samples) and converts it to the parquet format VeRL expects. Each row contains:

-   `prompt`: the math question (as a chat message)
-   `reward_model.ground_truth`: the correct numerical answer
-   `data_source`: dataset identifier

The reward function will compare the model's generated answer against the ground truth — correct answer gets reward 1.0, incorrect gets 0.0.

### Submit the VeRL GRPO training job

Now we submit the RL training job. In the Jupyter terminal, run:

``` bash
# run in a terminal inside jupyter container, from inside the "work" directory
cd ~/work
ray job submit --address="http://ray-head:8265" \
  --runtime-env runtime.json \
  --working-dir . \
  -- python3 -m verl.trainer.main_ppo \
  algorithm.adv_estimator=grpo \
  data.train_files=data/gsm8k/train.parquet \
  data.val_files=data/gsm8k/test.parquet \
  data.train_batch_size=8 \
  data.max_prompt_length=256 \
  data.max_response_length=128 \
  data.filter_overlong_prompts=True \
  data.truncation=error \
  actor_rollout_ref.model.path=Qwen/Qwen2.5-0.5B-Instruct \
  actor_rollout_ref.rollout.name=hf \
  actor_rollout_ref.rollout.tensor_model_parallel_size=1 \
  actor_rollout_ref.rollout.n=2 \
  actor_rollout_ref.rollout.top_k=0 \
  actor_rollout_ref.actor.optim.lr=1e-6 \
  actor_rollout_ref.actor.ppo_mini_batch_size=8 \
  actor_rollout_ref.actor.ppo_micro_batch_size_per_gpu=4 \
  actor_rollout_ref.actor.use_kl_loss=True \
  actor_rollout_ref.actor.kl_loss_coef=0.001 \
  actor_rollout_ref.model.enable_gradient_checkpointing=True \
  actor_rollout_ref.ref.log_prob_micro_batch_size_per_gpu=4 \
  algorithm.use_kl_in_reward=False \
  trainer.n_gpus_per_node=1 \
  trainer.nnodes=2 \
  trainer.total_epochs=1 \
  trainer.save_freq=-1 \
  trainer.test_freq=1 \
  trainer.val_before_train=False \
  trainer.logger='["console"]'
```

Key parameters:

-   `algorithm.adv_estimator=grpo` — use GRPO (no critic model needed)
-   `actor_rollout_ref.model.path=Qwen/Qwen2.5-0.5B-Instruct` — smallest Qwen model (0.5B parameters)
-   `actor_rollout_ref.rollout.name=hf` — use HuggingFace native generation (compatible with AMD MI100 GPUs)
-   `actor_rollout_ref.rollout.n=2` — generate 2 responses per prompt for GRPO comparison
-   `trainer.n_gpus_per_node=1, trainer.nnodes=2` — use both GPU workers in the cluster
-   `trainer.total_epochs=1` — single epoch for a quick smoke test

The first run will download the Qwen2.5-0.5B-Instruct model from HuggingFace, which may take a few minutes.

While it is running, check the Ray dashboard:

-   In the "Jobs" tab, you should see the job go from PENDING to RUNNING
-   In the "Overview" tab, observe the GPU resource usage
-   Click on the job to see its logs

Wait for the job to finish (SUCCEEDED state).

### Stop the Ray cluster

When you are finished, stop the cluster:

For AMD GPUs:

``` bash
# run on node-mltrain
docker compose -f mltrain-chi/docker/docker-compose-ray-rocm.yaml down
```

And stop the Jupyter server:

``` bash
# run on node-mltrain
docker stop jupyter
```